# Intro to Python — Hands-On Uygulama (CEVAP ANAHTARI)
**Tema: X Bankası Kredi Başvuruları (2023–2024)**

X Bankası'nın kredi başvuru sistemi, 2023–2024 arasında alınan başvuruları tek bir dosyaya (`credit_applications_raw.csv`) aktarmış. Ancak dosya hiç temizlenmemiş: kolon isimlerinde boşluklar var, tutarlar `"$12,500.00"` gibi string olarak yazılmış, tarihler iki farklı formatta karışık, bazı satırlar tekrar ediyor ve bazı kategori değerleri tutarsız yazılmış (`APPROVED` / `Approved` / `approved` gibi).

Bu notebook, lecture'da öğrendiğiniz adımları uygulayarak bu dosyayı **temiz, analiz edilebilir bir veri setine** dönüştürüyor. Bu, **cevap anahtarı** versiyonudur — tüm kod tamamlanmış haldedir.

Süre: ~45 dakika (kendiniz çalışırken)


### 0 · Veri dosyalarını indir
Önce bu hücreyi çalıştırın: veri dosyaları GitHub'dan bu oturuma iner.

In [ ]:
import os, urllib.request
BASE = "https://raw.githubusercontent.com/ardaergene35/python-intro/main/practice/"
for f in ["credit_applications_raw.csv", "loan_purpose_lookup.csv"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + f, f)
    print('Hazır:', f)

## 1 · Kütüphaneler ve görüntü ayarları

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("pandas:", pd.__version__, "| numpy:", np.__version__)


## 2 · Dosyayı yüklemeden önce bir göz atın

Önce `nrows=5` ile hızlı bir önizleme yapıyoruz, sonra `na_values` ve `skipinitialspace` ile tam veriyi yüklüyoruz.

In [ ]:
preview = pd.read_csv("credit_applications_raw.csv", nrows=5)
preview


In [ ]:
df = pd.read_csv(
    "credit_applications_raw.csv",
    skipinitialspace=True,
    na_values=["", " ", "NA", "N/A"],
)
print("Shape:", df.shape)
df.head(3)


## 3 · İlk bakış: sample, info, memory

In [ ]:
df.sample(5, random_state=1)


In [ ]:
df.info()


In [ ]:
df.memory_usage(deep=True) / 1024   # KB


## 4 · Kolon isimlerini düzeltin

Kolon isimlerinde baştaki/sondaki boşluklar ve büyük harfler var. `.str.strip().str.lower()` ile normalize ediyoruz.

In [ ]:
print("Before:", df.columns.tolist())

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
)

df = df.rename(columns={"applicant id": "applicant_id"})

print("After :", df.columns.tolist())


## 5 · Dtype düzeltme — `requested_amount`

Kolon `"$12,500.00"` formatında string. Önce `$` ve `,` karakterlerini temizleyip sonra `pd.to_numeric(errors="coerce")` ile sayıya çeviriyoruz.

In [ ]:
df["requested_amount"] = (
    df["requested_amount"].astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
)
df["requested_amount"] = pd.to_numeric(df["requested_amount"], errors="coerce")

print("dtype:", df["requested_amount"].dtype)
print("Cevirim sonrasi NaN sayisi:", df["requested_amount"].isna().sum())
df["requested_amount"].describe()


## 6 · Dtype düzeltme — `application_date`

Tarihler iki farklı formatta karışık geldi (`dd/mm/yyyy` ve `yyyy-mm-dd`). `format="mixed"` bunu tek satırda çözer.

In [ ]:
df["application_date"] = pd.to_datetime(df["application_date"], format="mixed", dayfirst=True, errors="coerce")
print("dtype:", df["application_date"].dtype)
print("Range:", df["application_date"].min(), "->", df["application_date"].max())


## 7 · Eksik değerler (missing values)

Önce `dropna=False` ile gizli NaN'ları görün, sonra `channel` ve `city` kolonlarını `"UNKNOWN"` ile doldurun.

In [ ]:
df["channel"].value_counts(dropna=False)


In [ ]:
(df.isna().mean() * 100).round(2).sort_values(ascending=False)


In [ ]:
df["channel"] = df["channel"].fillna("UNKNOWN")
df["city"] = df["city"].fillna("UNKNOWN")

df[["channel", "city"]].isna().sum()


## 8 · Tekrar eden (duplicate) satırlar

Tam satır duplicate'leri bulup çıkarıyoruz.

In [ ]:
print("Full-row duplicates:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6)


In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows. Now: {len(df)}")


## 9 · Kategorik temizlik

`status` kolonu `APPROVED` / `Approved` / `approved` gibi tutarsız yazılmış. `.str.strip().str.upper()` ile normalize edip düşük kardinaliteli kolonları `category` dtype'a çeviriyoruz.

In [ ]:
df["status"].value_counts(dropna=False)


In [ ]:
df["status"] = df["status"].str.strip().str.upper()
df["city"] = df["city"].str.strip().str.upper()

print(df["status"].value_counts(dropna=False))
print()
print(df["city"].value_counts(dropna=False))


In [ ]:
for col in ["city", "channel", "status"]:
    df[col] = df[col].astype("category")

df.dtypes


## 10 · Filtreleme

Üç farklı filtreleme yöntemini kullanıyoruz: boolean mask, `.loc[]`, `.query()`. Ayrıca `&`, `|`, `~` operatörlerini birleştiriyoruz (`and`/`or` DEĞİL!).

In [ ]:
high_value = df[df["requested_amount"] > 100000]
print("requested_amount > 100000 olan basvuru sayisi:", len(high_value))


In [ ]:
istanbul_high = df.query("city == 'ISTANBUL' and requested_amount > 50000")
print("Istanbul & requested_amount > 50000:", istanbul_high.shape[0])


In [ ]:
approved_online = df[(df["status"] == "APPROVED") & (df["channel"] == "Online")]
print("APPROVED & Online basvuru sayisi:", len(approved_online))


## 11 · Keşifsel Veri Analizi (EDA)

`describe()`, `value_counts(normalize=True)`, `crosstab`, `pivot_table`.

In [ ]:
df["requested_amount"].describe()


In [ ]:
df["city"].value_counts(normalize=True).mul(100).round(2)


In [ ]:
pd.crosstab(df["city"], df["status"], normalize="index").mul(100).round(1)


In [ ]:
pd.pivot_table(
    df,
    index="city",
    columns="channel",
    values="requested_amount",
    aggfunc="mean",
    observed=True,
).round(2)


## 12 · Capping, binning, groupby ve merge

- `.clip()` ile p99 üstündeki uç değerleri sınırlıyoruz (winsorization)
- `groupby().agg()` ile şehir bazlı özet çıkarıyoruz
- `loan_purpose_code` kolonunu bir lookup tablosuyla `merge` edip (`indicator=True`) eşleşmeyenleri buluyoruz

In [ ]:
cap99 = df["requested_amount"].quantile(0.99)
df["amount_capped"] = df["requested_amount"].clip(lower=0, upper=cap99)
print("p99 cap value:", round(cap99, 2))
df[["requested_amount", "amount_capped"]].describe(percentiles=[0.95, 0.99])


In [ ]:
df["amount_decile"] = pd.qcut(df["amount_capped"], q=10, labels=False, duplicates="drop")
df.groupby("amount_decile", observed=True)["amount_capped"].agg(["count", "min", "max"])


In [ ]:
summary = (
    df.groupby("city", observed=True)
      .agg(
          basvuru_sayisi=("requested_amount", "size"),
          ortalama_tutar=("amount_capped", "mean"),
          onay_orani=("status", lambda s: (s == "APPROVED").mean()),
      )
      .sort_values("basvuru_sayisi", ascending=False)
)
summary


In [ ]:
lookup = pd.read_csv("loan_purpose_lookup.csv")
df = df.merge(lookup, on="loan_purpose_code", how="left", indicator=True)
print(df["_merge"].value_counts())

# Eslesmeyen kodlar hangileri?
df.loc[df["_merge"] == "left_only", "loan_purpose_code"].value_counts(dropna=False)


In [ ]:
df = df.drop(columns="_merge")
df["loan_purpose_group"].value_counts(dropna=False)


## 13 · Basit görselleştirmeler

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(df["amount_capped"], bins=50, kde=True, ax=ax)
ax.set_title("Talep edilen kredi tutari dagilimi (p99 capped)")
ax.set_xlabel("Tutar (TL)")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df, x="channel", y="amount_capped", ax=ax)
ax.set_title("Kanala gore talep tutari")
plt.tight_layout()
plt.show()


## 14 · Google Form — Sonuçlarınızı girin

Yukarıdaki adımları tamamladıysanız artık aşağıdaki soruların cevaplarına sahipsiniz. Formda karşınıza çıkacak sorular bu notebook'ta ürettiğiniz sayılarla birebir eşleşiyor:

- Ham veri kaç satır?
- `requested_amount` çevriminde kaç NaN oluştu?
- `channel` ve `city` kolonlarında kaç eksik değer var?
- Kaç tam satır duplicate bulundu, temizlik sonrası kaç satır kaldı?
- `requested_amount > 100000` olan kaç başvuru var?
- İstanbul'da `requested_amount > 50000` olan kaç başvuru var?
- `APPROVED` & `Online` olan kaç başvuru var?
- `requested_amount` medyanı ve ortalaması nedir?
- Hangi şehir en yüksek paya sahip, payı yüzde kaç?
- İstanbul'un onay oranı yüzde kaç?
- p99 cap değeri nedir?
- Genel onay oranı yüzde kaç?
- Kaç adet `loan_purpose_code` lookup tablosunda eşleşmedi?
- En çok görülen kredi amacı (`loan_purpose_group`) nedir?
